In [30]:
"""
    "U1234567A",
    "U7654321B",
    "U2345678C",
    "U8765432D",
    "U3456789E",
    "U9876543F",
    "U4567890G",
    "U0987654H",
    "U5678901I",
    "U1098765J",
    "U6789012K",
    "U2109876L",
    "U7890123M",
    "U3210987N",
    "U8901234O",
    "U4321098P",
    "U9012345Q",
    "U5432109R",
    "U1122334S",
    "U9988776T",
    "U2340793K",
    "U2340246H",
    "U2221633B",
"""
MATRIC_NUMBERS = [
    "U2240731L"
]

# Path to your CSV file (if notebook is in same folder as CSV, leave as is)
CSV_PATH = "ResalePricesSingapore.csv"

In [31]:
# ============================================================
# CELL 2 — Digit-to-Town mapping (from project spec Table 1)
# ============================================================

DIGIT_TO_TOWN = {
    '0': 'BEDOK',
    '1': 'BUKIT PANJANG',
    '2': 'CLEMENTI',
    '3': 'CHOA CHU KANG',
    '4': 'HOUGANG',
    '5': 'JURONG WEST',
    '6': 'PASIR RIS',
    '7': 'TAMPINES',
    '8': 'WOODLANDS',
    '9': 'YISHUN'
}

# Year last digit to actual year (data covers 2015-2024, not 2025)
DIGIT_TO_YEAR = {
    '5': 2015, '6': 2016, '7': 2017, '8': 2018, '9': 2019,
    '0': 2020, '1': 2021, '2': 2022, '3': 2023, '4': 2024
}

def decode_matric(matric):
    """
    Given a matric number like U2340246H, extract:
    - target year   (last digit of number portion)
    - start month   (second last digit; '0' = October)
    - towns         (all unique digits in matric)
    """
    # Extract only the digit characters
    digits = [c for c in matric if c.isdigit()]

    last_digit        = digits[-1]   # target year
    second_last_digit = digits[-2]   # start month

    target_year  = DIGIT_TO_YEAR[last_digit]
    start_month  = 10 if second_last_digit == '0' else int(second_last_digit)
    towns        = list(dict.fromkeys(  # unique digits, preserve order
                       DIGIT_TO_TOWN[d] for d in digits if d in DIGIT_TO_TOWN
                   ))

    return target_year, start_month, towns


# Preview what each matric decodes to
print(f"{'Matric':<15} {'Year':<6} {'Start Month':<13} Towns")
print("-" * 75)
for m in MATRIC_NUMBERS:
    y, sm, t = decode_matric(m)
    print(f"{m:<15} {y:<6} {sm:<13} {', '.join(t)}")

Matric          Year   Start Month   Towns
---------------------------------------------------------------------------
U2240731L       2021   3             CLEMENTI, HOUGANG, BEDOK, TAMPINES, CHOA CHU KANG, BUKIT PANJANG


In [32]:
# ============================================================
# CELL 3 — Load the data (runs once)
# ============================================================
import pandas as pd

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().upper() for c in df.columns]
df = df.rename(columns={'FLOOR_AREA_SQM': 'FLOOR_AREA'})

# Parse month column e.g. 'Apr-16' → year=2016, month=4
df['PARSED_DATE'] = pd.to_datetime(df['MONTH'], format='%b-%y')
df['YEAR']        = df['PARSED_DATE'].dt.year
df['MONTH_NUM']   = df['PARSED_DATE'].dt.month

print(f"✅ Loaded {len(df):,} rows")
print(f"   Year range: {df['YEAR'].min()} – {df['YEAR'].max()}")

✅ Loaded 259,237 rows
   Year range: 2015 – 2025


In [33]:
# ============================================================
# CELL 4 — Query function
# ============================================================

PRICE_THRESHOLD = 4725

def run_query(matric, df):
    """
    For a given matric number, run all (x, y) combinations and
    return a DataFrame of valid results.
    """
    target_year, start_month, towns = decode_matric(matric)
    results = []

    for x in range(1, 9):           # x = 1 to 8
        end_month_num = start_month + x - 1

        # Handle year rollover (e.g. start Oct, x=4 → goes into next year)
        if end_month_num <= 12:
            end_year  = target_year
            end_month = end_month_num
        else:
            end_year  = target_year + 1
            end_month = end_month_num - 12

        # Filter by time window
        if end_year == target_year:
            mask_time = (
                (df['YEAR']      == target_year) &
                (df['MONTH_NUM'] >= start_month) &
                (df['MONTH_NUM'] <= end_month)
            )
        else:
            # Spans two years
            mask_time = (
                ((df['YEAR'] == target_year) & (df['MONTH_NUM'] >= start_month)) |
                ((df['YEAR'] == end_year)    & (df['MONTH_NUM'] <= end_month))
            )

        # Filter by towns (done once per x, not per y — saves time)
        df_x = df[mask_time & df['TOWN'].isin(towns)].copy()

        for y in range(80, 151):     # y = 80 to 150
            df_xy = df_x[(df_x['FLOOR_AREA'] >=y)  & (df_x['RESALE_PRICE']>0) & df_x['FLOOR_AREA']>0].copy()

            if df_xy.empty:
                continue 
            df_xy['PRICE_PER_SQM'] = df_xy['RESALE_PRICE'] / df_xy['FLOOR_AREA']
            min_ppsm = df_xy['PRICE_PER_SQM'].min()

            if min_ppsm > PRICE_THRESHOLD:
                continue

            row = df_xy.loc[df_xy['PRICE_PER_SQM'].idxmin()]

            results.append({
                '(x, y)':                f'({x}, {y})',
                'Year':                  int(row['YEAR']),
                'Month':                 f"{int(row['MONTH_NUM']):02d}",
                'Town':                  row['TOWN'],
                'Block':                 row['BLOCK'],
                'Floor_Area':            int(row['FLOOR_AREA']),
                'Flat_Model':            row['FLAT_MODEL'],
                'Lease_Commence_Date':   int(row['LEASE_COMMENCE_DATE']),
                'Price_Per_Square_Meter': round(min_ppsm)
            })

    return pd.DataFrame(results)

print("Query function ready")

Query function ready


In [34]:
# ============================================================
# CELL 5 — Run for ALL matric numbers and save CSVs
# ============================================================

for matric in MATRIC_NUMBERS:
    print(f"\n⏳ Processing {matric}...")
    year, month, towns = decode_matric(matric)
    print(f"   Year: {year}, Start Month: {month}, Towns: {', '.join(towns)}")

    result_df = run_query(matric, df)

    filename = f"ScanResult_{matric}.csv"
    result_df.to_csv(filename, index=False)

    print(f"   ✅ {len(result_df)} valid (x,y) pairs → saved as {filename}")
    print(result_df.head(5).to_string())

print("\ndone")


⏳ Processing U2240731L...
   Year: 2021, Start Month: 3, Towns: CLEMENTI, HOUGANG, BEDOK, TAMPINES, CHOA CHU KANG, BUKIT PANJANG
   ✅ 568 valid (x,y) pairs → saved as ScanResult_U2240731L.csv
    (x, y)  Year Month           Town Block  Floor_Area Flat_Model  Lease_Commence_Date  Price_Per_Square_Meter
0  (1, 80)  2021    03  BUKIT PANJANG   212         104    Model A                 1988                    3269
1  (1, 81)  2021    03  BUKIT PANJANG   212         104    Model A                 1988                    3269
2  (1, 82)  2021    03  BUKIT PANJANG   212         104    Model A                 1988                    3269
3  (1, 83)  2021    03  BUKIT PANJANG   212         104    Model A                 1988                    3269
4  (1, 84)  2021    03  BUKIT PANJANG   212         104    Model A                 1988                    3269

done
